This project requires Python 3.7 or above:

In [24]:
import sys
print(sys.executable)

assert sys.version_info >= (3, 7)

/usr/bin/python3


In [5]:
# Is this notebook running on Colab or Kaggle?
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

if IS_COLAB or IS_KAGGLE:
    %pip uninstall -y gym  # to avoid any conflicts with Gymnasium
    %pip install -q box2d

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 87.9 MB/s eta 0:00:00:00:01


In [10]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [25]:
!/usr/bin/python3 -m pip install --user gymnasium[atari] autorom



In [29]:
!/usr/bin/python3 -m pip show autorom
!/usr/bin/python3 -m autorom.accept_rom_license

Name: AutoROM
Version: 0.6.1
Summary: Automated installation of Atari ROMs for Gym/ALE-Py
Home-page: https://github.com/Farama-Foundation/AutoROM
Author: Farama Foundation
Author-email: contact@farama.org
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: click, requests
Required-by: 
/usr/bin/python3: Error while finding module specification for 'autorom.accept_rom_license' (ModuleNotFoundError: No module named 'autorom')


In [39]:
import gymnasium as gym

# Create the Pong environment
env = gym.make("ALE/Pong-v5", render_mode="human")

# Reset the environment to get the initial observation
observation, info = env.reset()

# Print the shape of the observation space
print(f"Observation space shape: {env.observation_space.shape}")

# Print the number of possible actions
print(f"Number of actions: {env.action_space.n}")

Observation space shape: (210, 160, 3)
Number of actions: 6


In [ ]:
import gymnasium as gym
import numpy as np
import os
from PIL import Image
from collections import defaultdict
from gymnasium.wrappers import RecordVideo

def preprocess_obs(obs_rgb, size=(16, 16), threshold=100):
    """
    obs_rgb: np.ndarray (H, W, 3), uint8
    1) Grayscale
    2) Resize to 'size'
    3) Binarize with threshold
    Returns a tuple of 0/1 ints length size[0]*size[1]
    """
    img = Image.fromarray(obs_rgb)
    img = img.convert("L")  # grayscale
    img = img.resize(size, Image.BILINEAR)
    arr = np.array(img, dtype=np.uint8)
    binarized = (arr > threshold).astype(np.uint8)
    return tuple(int(v) for v in binarized.flatten())

def q_values_for_state(Q, state, n_actions):
    return np.array([Q[(state, a)] for a in range(n_actions)], dtype=np.float32)

env = gym.make("ALE/Pong-v5", render_mode="rgb_array")  # avoid render_mode="human" for speed

video_dir = "./videos"
os.makedirs(video_dir, exist_ok=True)

env = RecordVideo(env, video_folder=video_dir, episode_trigger=lambda ep: True)

n_actions = env.action_space.n
print("Actions:", n_actions)

# Hyperparameters
alpha = 0.1
gamma = 0.99
epsilon = 0.2
epsilon_min = 0.01
epsilon_decay = 0.995
episodes = 1000
max_steps = 10000

Q = defaultdict(float)
rng = np.random.default_rng(0)

for ep in range(1, episodes + 1):
    obs, info = env.reset()
    state = preprocess_obs(obs, size=(16, 16), threshold=100)
    done = False
    total_reward = 0.0
    steps = 0
    frame_count = 0

    while not done and steps < max_steps:
        # Epsilon-greedy
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            q_vals = q_values_for_state(Q, state, n_actions)
            action = int(np.argmax(q_vals))

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1

        next_state = preprocess_obs(next_obs, size=(16, 16), threshold=100)
        frame_count += 1
        # Q-learning update
        current_q = Q[(state, action)]
        next_max_q = 0.0 if done else np.max(q_values_for_state(Q, next_state, n_actions))
        target = reward + gamma * next_max_q
        Q[(state, action)] = (1 - alpha) * current_q + alpha * target

        state = next_state
    print("Frames recorded:", frame_count)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    print(f"Episode {ep:03d} | Return: {total_reward:.1f} | Epsilon: {epsilon:.3f} | frame: {frame_count}")

env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Actions: 6
Frames recorded: 842
Episode 001 | Return: -20.0 | Epsilon: 0.199
Frames recorded: 764
Episode 002 | Return: -21.0 | Epsilon: 0.198
Frames recorded: 1054
Episode 003 | Return: -19.0 | Epsilon: 0.197
Frames recorded: 792
Episode 004 | Return: -21.0 | Epsilon: 0.196
Frames recorded: 824
Episode 005 | Return: -21.0 | Epsilon: 0.195
Frames recorded: 914
Episode 006 | Return: -21.0 | Epsilon: 0.194
Frames recorded: 826
Episode 007 | Return: -21.0 | Epsilon: 0.193
Frames recorded: 988
Episode 008 | Return: -20.0 | Epsilon: 0.192
Frames recorded: 902
Episode 009 | Return: -20.0 | Epsilon: 0.191
Frames recorded: 792
Episode 010 | Return: -21.0 | Epsilon: 0.190
Frames recorded: 826
Episode 011 | Return: -21.0 | Epsilon: 0.189
Frames recorded: 1004
Episode 012 | Return: -21.0 | Epsilon: 0.188
Frames recorded: 792
Episode 013 | Return: -21.0 | Epsilon: 0.187
Frames recorded: 1012
Episode 014 | Return: -19.0 | Epsilon: 0.186
Frames recorded: 820
Episode 015 | Return: -21.0 | Epsilon: 0.

In [58]:

import glob, os
video_dir = "./videos"
files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
print("Videos:", files)
for f in files:
    print(f, os.path.getsize(f), "bytes")


Videos: ['./videos/rl-video-episode-0.mp4', './videos/rl-video-episode-1.mp4', './videos/rl-video-episode-10.mp4', './videos/rl-video-episode-11.mp4', './videos/rl-video-episode-12.mp4', './videos/rl-video-episode-13.mp4', './videos/rl-video-episode-14.mp4', './videos/rl-video-episode-15.mp4', './videos/rl-video-episode-16.mp4', './videos/rl-video-episode-17.mp4', './videos/rl-video-episode-18.mp4', './videos/rl-video-episode-19.mp4', './videos/rl-video-episode-2.mp4', './videos/rl-video-episode-20.mp4', './videos/rl-video-episode-21.mp4', './videos/rl-video-episode-22.mp4', './videos/rl-video-episode-23.mp4', './videos/rl-video-episode-24.mp4', './videos/rl-video-episode-25.mp4', './videos/rl-video-episode-26.mp4', './videos/rl-video-episode-27.mp4', './videos/rl-video-episode-28.mp4', './videos/rl-video-episode-29.mp4', './videos/rl-video-episode-3.mp4', './videos/rl-video-episode-30.mp4', './videos/rl-video-episode-31.mp4', './videos/rl-video-episode-32.mp4', './videos/rl-video-epis

In [60]:

from google.colab import files
files.download("./videos/rl-video-episode-0.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:

from IPython.display import Video
Video("./videos/rl-video-episode-99.mp4")
